# Lesson 4.1 — The Motion Stack

Execute turns a validated reference into real joint motion: sample the reference, track it closed-loop with Module 8's controller, drive the plant. We run the real stack and confirm it reaches the target.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception,
    understand, to_configuration, plan_reference, execute_reference,
    velocity_layer, geometric_jacobian, fk_xy, P2, T2)
checks = []
world = GreenhouseWorld.demo_row(n=6, seed=1)
dets = model_perception(world, rng=np.random.default_rng(7))
_, tgt = understand(dets, world)
layer = plan_reference(world.q.copy(), to_configuration(tgt), rng=np.random.default_rng(0))
print('planning F3:', tgt['id'], '| validated:', layer['validated'])


planning F3: F3 | validated: True


### Run the closed loop: sample -> control -> drive

In [2]:
rec = execute_reference(layer)
final_tool = fk_xy(P2, T2, rec['q'][-1])
print('reached:', rec['reached'], '| tracking rms: %.5f' % rec['rms'])
print('final tool:', np.round(final_tool,3), '| target:', np.round(tgt['xy'],3))
checks.append(rec['reached'])                                   # converged to target
checks.append(rec['rms'] < 0.02)                                # tight tracking
checks.append(np.linalg.norm(final_tool - tgt['xy']) < 1e-2)    # FK lands on the fruit


reached: True | tracking rms: 0.00007
final tool: [0.447 0.152] | target: [0.447 0.152]


### Feedback recovers from a disturbance an open loop could not

In [3]:
def kick(t, j): return 30.0 if (j == 0 and 0.4 < t < 0.5) else 0.0
rec_d = execute_reference(layer, disturbance=kick)
print('with disturbance -> reached:', rec_d['reached'], '| rms: %.4f' % rec_d['rms'])
checks.append(rec_d['reached'])   # feedback drives the joint back to the reference
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


with disturbance -> reached: True | rms: 0.1114
4/4 checks passed.
All checks passed.
